# Cleaning the job ads

The collection in `01_collect.ipynb` produced 3,570 ads with full texts from two
snapshots (7 October and 19 November 2026). Before
skills can be extracted, the data needs work: some ads are duplicates, some
fields contain nested structures, and some ads may not be data roles at all.

This notebook goes through these problems one at a time and saves a clean table
to `data/processed/`. Like the raw data, it is not published on GitHub.

## Setup

The ads come from two collection days (see `01_collect.ipynb`): 7 October and
19 November 2026. The full texts are in `details.jsonl`, one file per snapshot.
An ad that was online on both days has only one text, from the day it was first
collected; that day is stored in the new column `collected`.

The search terms that found each ad are only stored in `search_results.json`,
but they are needed later to compare roles, so these files are loaded too.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path.cwd().parent / "data" / "raw"
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# two collection days: the first snapshot in data/raw/, the second in its own folder
SNAPSHOTS = {"2026-10-07": RAW_DIR, "2026-11-19": RAW_DIR / "2026-11-19"}

details, search = [], []
for date, folder in SNAPSHOTS.items():
    d = pd.read_json(folder / "details.jsonl", lines=True)
    d["collected"] = pd.Timestamp(date)   # the day the ad's text was collected
    details.append(d)

    s = pd.DataFrame(json.loads((folder / "search_results.json").read_text(encoding="utf-8")))
    s["snapshot"] = date
    search.append(s[["referenznummer", "suchbegriffe", "snapshot"]])

details = pd.concat(details, ignore_index=True)
search = pd.concat(search, ignore_index=True)

print(details.shape)
print(details["collected"].value_counts().sort_index(), "\n")
print(details["referenznummer"].duplicated().sum(), "ads with a text in both snapshots")
print(search["referenznummer"].nunique(), "unique ads in the search results of both snapshots")

## Overview: dtypes and missing values

All 42 fields (plus the new column `collected`) have been loaded into the `details` DataFrame. Now we can inspect what data types pandas has inferred for each, and how often values are missing.


In [ ]:
overview = pd.DataFrame({
    "dtypes": details.dtypes,
    "missing": details.isna().sum(),
    "missing_%": (details.isna().mean() * 100).round(1)
})

overview.sort_values("missing", ascending=False)

**The fields the analysis needs are complete.** The description, the title, the
main occupation (`hauptberuf`), the locations and the publication date are
filled in every ad. The employer (`firma`) is missing in only one ad.

**27 of the 42 fields have gaps, and some are almost empty:**

- **Salary.** Only 460 ads (13%) give a salary range (`gehaltsspanneVon`/`Bis`),
  and only 68 a fixed salary (`festgehalt`). Salary can therefore not be
  analysed reliably and is not one of the project's questions.
- **Contract details.** `befristetBis`, `befristungInMonaten` and
  `homeofficeprozent` are missing in over 94% of the ads. The complete field
  `vertragsdauer` covers the contract type instead.
- **Technical fields.** `chiffrenummer`, `externeURL`, `arbeitgeberKundennummerHash`
  and the `allianzpartner` fields describe how the ad was published, not the job.

**The yes/no fields have a different kind of gap.** `homeofficemoeglich` is
empty in 44% of the ads, the part-time fields in about a third. An empty field
probably means "not specified" rather than "no", so it cannot simply be filled
with 0. The next cell checks what values these fields actually contain.

**Two date fields are stored as text** (`datumErsteVeroeffentlichung`,
`aenderungsdatum`) and need converting to dates.

In [ ]:
for col in ["verguetungsangabe", "vertragsdauer", "stellenangebotsart",
            "homeofficemoeglich", "arbeitszeitVollzeit"]:
    print(details[col].value_counts(dropna=False), "\n")

**`stellenangebotsart` has only one value** (`ARBEIT`, regular employment) — the
filter `angebotsart = 1` in the collection worked. The field carries no
information and is dropped.

**`vertragsdauer` is usable, with its own "no information" category:** 58% of
the ads are permanent, 7% fixed-term, and 35% say nothing. Salary works the same
way: `verguetungsangabe` shows that 3,042 ads (85%) give no salary information.

**The yes/no fields have three states, not two.** `homeofficemoeglich` contains
`1.0` (612 ads), `0.0` (1,399 ads) and empty (1,559 ads). Since an explicit "no"
exists, an empty field means "not specified", not "no". This matters for the
numbers: home office is offered in 17% of all ads, but in 30% of the ads that
say anything about it. The empty fields are therefore kept as missing, not filled with 0.

## Selecting the columns for analysis

The project's questions are about skills, roles and seniority, so the analysis
needs the text, the title, the occupation and a few descriptive fields. Fields
that are almost empty, constant or purely technical are dropped.

`istPrivateArbeitsvermittlung` (posted by a recruitment agency) is kept for now:
the duplicates seen during the collection came from an agency, and this field
can show whether that is a pattern.

The search terms that found each ad are added from the search results, so that
roles can be compared later.

In [ ]:
keep = [
    "referenznummer", "stellenangebotsTitel", "firma", "hauptberuf",
    "stellenangebotsBeschreibung", "stellenlokationen",
    "datumErsteVeroeffentlichung", "vertragsdauer",
    "arbeitszeitVollzeit", "homeofficemoeglich", "istPrivateArbeitsvermittlung",
    "collected",
]

jobs = details[keep].copy()

# add the search terms from both snapshots, and in how many snapshots the ad was online
terms = (search.explode("suchbegriffe")
               .groupby("referenznummer")
               .agg(suchbegriffe=("suchbegriffe", lambda t: list(dict.fromkeys(t))),
                    n_snapshots=("snapshot", "nunique"))
               .reset_index())
jobs = jobs.merge(terms, on="referenznummer", how="left")

print(jobs.shape)
print(jobs["suchbegriffe"].isna().sum(), "ads without search terms")
jobs.head()

The table now has 14 columns: the 12 selected fields (including `collected`),
`suchbegriffe`, and `n_snapshots`, the number of snapshots in which the search
found the ad. Every ad found its search terms, so the files match completely.

The first rows show the Berlin pair again (rows 3 and 4): same agency, same
title, same text, different postcodes — and also different publication dates
(14 September and 4 August). Both are flagged as posted by a private recruitment
agency (`istPrivateArbeitsvermittlung = 1.0`), and both were still online on
19 November (`n_snapshots = 2`). This is examined in the duplicates section.

## Fixing the boolean and date types

The yes/no fields are converted to pandas' `boolean` type, which can hold
True, False *and* a missing value, so "not specified" is kept. The publication
date is converted from text to a date.

In [ ]:
# convert yes/no fields to boolean
for col in ["arbeitszeitVollzeit", "homeofficemoeglich", "istPrivateArbeitsvermittlung"]:
    jobs[col] = jobs[col].astype("boolean")
    
# convert the publication date to datetime
jobs["datumErsteVeroeffentlichung"] = pd.to_datetime(jobs["datumErsteVeroeffentlichung"])

print(jobs.dtypes, "\n")
print(jobs["homeofficemoeglich"].value_counts(dropna=False), "\n")
print(jobs["datumErsteVeroeffentlichung"].min(), "to", jobs["datumErsteVeroeffentlichung"].max())

The conversion lost nothing: the home office counts are the same as before
(612 True, 1,399 False, 1,559 missing), and missing values are now shown as `<NA>`.

**The publication dates are a surprise, though.** The newest ad is from 19 November
2026, the day of the second collection. The oldest was first published on 31 March
2021 — five and a half years before the first collection, and still online. Such
ads may be positions that stay open for a long time, or "evergreen" postings that
are kept online to collect applications rather than to fill one specific job. The
next cell shows how many ads are this old.

The age of an ad is measured on the day its text was collected (`collected`), so
an ad from the second snapshot is not counted six weeks older than it was.

In [ ]:
# age on the day the ad was collected
age_days = (jobs["collected"] - jobs["datumErsteVeroeffentlichung"]).dt.days

print(age_days.describe(), "\n")

bins = [-1, 30, 90, 180, 365, 10_000]
labels = ["up to 1 month", "1-3 months", "3-6 months", "6-12 months", "over 1 year"]
print(pd.cut(age_days, bins=bins, labels=labels).value_counts(sort=False))

**Most ads are recent.** Half were first published in the last 22 days (the
median), and 84% in the last three months (2,203 + 813 of 3,570 ads). The median
is lower than in the first snapshot alone (34 days), because the second snapshot
only contains ads that were not yet online on 7 October: most of them are at most
six weeks old. The mean of 57 days is more than twice the median: a few very old
ads pull it up. The distribution is strongly skewed to the right.

**95 ads (2.7%) are older than a year**, the oldest over five years (2,016 days).

**Decision: all ads are kept.** Every one of them was online on the day of its
collection, so all of them are part of what employers currently advertise. Any
cutoff (six months? a year?) would be arbitrary. Instead, the age of each ad is
stored as a column, so the analysis can later check whether the old ads change
the results.

In [ ]:
# Insert the age of each ad as a new column.
jobs["age_days"] = age_days

jobs["age_days"].describe()

## Unpacking the locations

`stellenlokationen` holds a list of locations per ad, each a nested dictionary.
Before extracting the city and federal state, a look at its structure: how many
locations do ads have, and what does one entry contain?

In [ ]:
# Number of locations per ad
n_locations = jobs["stellenlokationen"].apply(len)
print(n_locations.value_counts().sort_index(), "\n")

jobs["stellenlokationen"].iloc[0]

**Most ads have a single location (3,175, or 89%), but 395 list several** — up
to ten. The count at exactly ten (52 ads) is higher than at nine (18 ads), which
suggests the job search may cut longer lists off at ten. If so, some ads are
advertised in even more places than the data shows.

Each location contains an address (street, house number, postcode, city,
federal state, country) and coordinates. Not every entry has every part: the
first ad has a street, others start with the postcode.

**How multi-location ads are handled:** one ad is one job posting, so for
counting skills it counts once, no matter how many locations it lists. For the
location columns, the first location is used, and the number of locations is
stored separately.

The address fields are taken from the first location. `.get()` returns `None`
instead of an error when a field is missing.

In [ ]:
def first_location(locations, key):
    """Return one address field of the first location, or None if it is missing."""
    if not locations:
        return None
    return locations[0].get("adresse", {}).get(key)

jobs["n_locations"] = n_locations
for key, name in [("plz", "postcode"), ("ort", "city"), ("region", "state"), ("land", "country")]:
    jobs[name] = jobs["stellenlokationen"].apply(first_location, key=key)

print(jobs[["postcode", "city", "state", "country"]].isna().sum(), "\n")
print(jobs["country"].value_counts(), "\n")
print(jobs["state"].value_counts())

**Not all ads are in Germany.** Based on the first location, 3,460 ads (96.9%)
are in Germany, but 110 are abroad: 98 in Austria, 6 in Switzerland and one
each in Slovakia, Israel, Spain, Italy, Portugal and Japan. The Bundesagentur's
job search also publishes ads from other countries.

All 16 German federal states appear, led by Bayern (773), Nordrhein-Westfalen
(639) and Baden-Württemberg (513). A few ads have no postcode (323), city (125)
or federal state (119); two have the state `SONSTIGES` (other).

**Decision: the analysis covers Germany only**, because the project asks what
*German* employers expect. One detail has to be checked first: the country
above comes from the first location only. An ad with several locations might
list a foreign location first and a German one later. Such an ad belongs in
the dataset.

In [ ]:
def has_german_location(locations):
    """True if at least one of the ad's locations is in Germany."""
    return any(loc.get("adresse", {}).get("land") == "DEUTSCHLAND" for loc in locations)

in_germany = jobs["stellenlokationen"].apply(has_german_location)
print(in_germany.value_counts(), "\n")

# ads that list Germany, but not as their first location
print(((jobs["country"] != "DEUTSCHLAND") & in_germany).sum(), "ads list Germany, but not first")

The two counts agree: 3,460 ads have a German location, and no ad lists Germany
only as a later location. So the first location is a reliable basis for the
location columns, and the filter can simply keep the ads in Germany.

In [ ]:
# Keep only ads with a German location and drop the country column
jobs = jobs[in_germany].drop(columns="country").reset_index(drop=True)

print(jobs.shape)
print(jobs[["postcode", "city", "state"]].isna().sum())

## Duplicates

The collection found exact duplicate descriptions: ads with different
reference numbers but identical text. Before deciding what to do with them, a
look at what these groups have in common: the same employer? the same title?
different locations?

In [ ]:
desc = jobs["stellenangebotsBeschreibung"]
dup = jobs[desc.duplicated(keep=False)]

print(desc.duplicated().sum(), "duplicate descriptions")
print(len(dup), "ads involved, sharing", dup["stellenangebotsBeschreibung"].nunique(), "different texts\n")

groups = (
    dup.groupby("stellenangebotsBeschreibung")
    .agg(
        n_ads=("referenznummer", "size"),
        employer=("firma", "first"),
        title=("stellenangebotsTitel", "first"),
        n_titles=("stellenangebotsTitel", "nunique"),
        n_cities=("city", "nunique"),
        n_postcodes=("postcode", "nunique"),
        agency=("istPrivateArbeitsvermittlung", "first"),
    )
    .sort_values("n_ads", ascending=False)
    .reset_index(drop=True)
)
groups

**After the Germany filter, 203 descriptions are duplicates** (5.9% of 3,460
ads), far more than in the first snapshot alone (32). They form 154 groups of
identical texts, 357 ads in total: 130 pairs, 18 groups of three, 2 of four,
3 of five, and one group of 20. In all but 7 groups the title is identical too;
in those 7, the titles differ only by a small addition, e.g. "AI Engineer (m/w/d)"
and "AI Engineer (m/w/d) 4 Tage Home-Office-Option", or a level ("Senior").

**Three kinds of duplicates:**

- **The same job posted again between the two snapshots (114 groups).** Same
  text, same location, but one copy is from 7 October and the other from
  19 November, under a new reference number. The job was published again,
  or its reference number changed. This is the main reason why there are many
  more duplicates now: the second snapshot only fetched ads with a new reference
  number, and some of them turned out to be old ads.
- **The same job in several places (22 groups).** The ads differ only in their
  location. PricewaterhouseCoopers posts one ad per city, e.g. "AI & Cloud
  Architect" in 20 cities; Staffbase lists three cities, and the Berlin pair from
  the collection has two postcodes. This is one job advertised for several
  locations.
- **The same job posted again on the same day (16 groups).** Same text, same
  title, same location, within one snapshot. For 2 more groups the location is
  missing, so the cases cannot be told apart.

**One employer stands out: Akkodis**, an engineering service provider, accounts
for 42 of the 154 groups, 36 of them across the two snapshots: Akkodis
republishes its ads regularly.

**Decision:** for counting skills, each text counts once, regardless of how many
copies or locations it has. Otherwise a job posted three times would count three
times. Before removing the copies, their number is stored in `n_copies`, so
this information is not lost. The copy that is kept is always the first one; for
an ad found on both days, this is the copy from 7 October.

**A finding for the next step:** many titles here are not data roles at all —
*Mechanical Design Engineer*, *Power Electronics Engineer*, *Senior Seat
Integration Engineer*, *Head of HR IT*. This confirms the suspicion from the
collection: broad search terms such as "Data Engineer" also bring in other
engineering and IT positions.

In [ ]:
jobs["n_copies"] = desc.map(desc.value_counts())
jobs = jobs.drop_duplicates(subset="stellenangebotsBeschreibung", keep="first").reset_index(drop=True)

print(jobs.shape)
print(jobs["n_copies"].value_counts().sort_index())

The numbers match: 3,460 − 203 = 3,257 ads remain, and `n_copies` records 130 ads
that had one copy, 18 that had two, 2 that had three, 3 that had four, and one
that had 19 (the PwC ad in 20 cities).

The check above only catches *identical* texts. If an employer changes a single
word, the copy is missed. A second, looser check: ads that share employer and
title but have different texts.

In [ ]:
same_job = jobs.duplicated(subset=["firma", "stellenangebotsTitel"], keep=False)
print(same_job.sum(), "ads share employer and title with another ad\n")

(jobs[same_job]
 .groupby(["firma", "stellenangebotsTitel"])
 .size()
 .sort_values(ascending=False)
 .head(15))

**275 ads share employer and title with another ad, but not their text.** The
largest groups come from a few employers: CHECK24 has two trainee programmes with
7 ads each, and KUMAVISION has seven job titles with 3 to 5 ads each. Whether these are the
same job with small text changes (for example the city named in the text) or
really different positions needs a look at the texts themselves. This is
checked in the next part.

### Near-duplicates: a closer look

To see what differs between ads with the same employer and title, two of the
CHECK24 trainee ads are compared line by line. `difflib` (from Python's
standard library) shows only the lines that differ, and a similarity score
between 0 (completely different) and 1 (identical).

In [ ]:
import difflib

group = jobs[(jobs["firma"] == "CHECK24 Services Personal GmbH")
             & (jobs["stellenangebotsTitel"] == "Associate Product Owner (m/w/d) Traineeprogramm")]
print(group[["city", "datumErsteVeroeffentlichung", "n_locations"]], "\n")

a, b = group["stellenangebotsBeschreibung"].iloc[:2]
print(f"similarity: {difflib.SequenceMatcher(None, a, b).ratio():.3f}\n")

# lines starting with - are only in the first text, lines with + only in the second
for line in difflib.unified_diff(a.splitlines(), b.splitlines(), lineterm="", n=0):
    print(line)


**The two texts differ in a single word: the city.** CHECK24 publishes one ad
per city (seven cities, all on 1 June 2026), and the only change is the
city name. The similarity is 0.999.

So these are multi-location postings, like the exact duplicates, but the
`duplicated()` check missed them because of one word. The next cell checks
whether this applies to all ads that share employer and title, or whether some
are genuinely different jobs.

In [ ]:
def similarity_to_first(texts):
    """Similarity of each text to the first text in its group (1 = identical)."""
    first = texts.iloc[0]
    return texts.apply(lambda t: difflib.SequenceMatcher(None, first, t).ratio())

same_job = jobs.duplicated(subset=["firma", "stellenangebotsTitel"], keep=False)
near = jobs[same_job].copy()
near["similarity"] = (near.groupby(["firma", "stellenangebotsTitel"])["stellenangebotsBeschreibung"]
                          .transform(similarity_to_first))

summary = (near.groupby(["firma", "stellenangebotsTitel"])["similarity"]
               .agg(n_ads="size", min_similarity="min")
               .sort_values("min_similarity"))
print(len(summary), "groups\n")
summary

**121 groups share employer and title. Their similarity has two clear ends and a
wider middle:**

- **Near-identical (59 groups, similarity 0.963 to 0.999).** The same text with
  minimal changes, like the CHECK24 example. These include all CHECK24 and
  KUMAVISION groups. 146 ads in total.
- **Genuinely different (28 groups, below 0.65).** The same title but a
  different text — for example, TÜV SÜD has two "Data Scientist" ads with a
  similarity of only 0.03. These are separate positions, probably in different
  teams, and are kept.

In between, 34 groups (69 ads) have a similarity between 0.68 and 0.95. These
could be the same job with larger edits — for example an ad that was revised
before it was published again in November — or similar jobs.

**Decision: ads with a similarity of 0.95 or higher are treated as copies.**
No group falls between 0.947 and 0.963, so the threshold still sits in a gap
above the uncertain middle zone, and only clear copies are removed. The gap is
narrower than in the first snapshot alone (where the middle zone ended at 0.94),
because some ads were edited between the two collection days. The middle zone is
kept: in the worst case, about 35 jobs are counted twice out of about 3,200
(1%), which cannot change the results in a noticeable way. As with the exact
duplicates, the number of copies is added to `n_copies` of the ad that is kept.

In [ ]:
THRESHOLD = 0.95
key = ['firma', 'stellenangebotsTitel']

# mark near-duplicate ads as copies
near["is_copy"] = (near["similarity"] >= THRESHOLD) & near.duplicated(subset=key)

# sum the number of copies for each group of near-duplicate ads
extra = near[near["is_copy"]].groupby(key)["n_copies"].sum()
keepers = near[~near.duplicated(subset=key)]

# map the number of extra copies to the keepers
add = keepers.set_index(key).index.map(lambda k: extra.get(k, 0))

# update the n_copies column of the keepers with the mapped extra copies
jobs.loc[keepers.index, "n_copies"] += add.values

# drop the near-duplicate ads that have been marked as copies
jobs = jobs.drop(index=near.index[near["is_copy"]]).reset_index(drop=True)

print(near["is_copy"].sum(), "near-copies removed")
print(jobs.shape)
print(jobs["n_copies"].value_counts().sort_index())


87 near-copies were removed (42 within the first snapshot, 45 from the second),
leaving 3,170 ads. The counts confirm that nothing was lost: adding up
`n_copies` over all ads (2,960 × 1 + 171 × 2 + 24 × 3 + 8 × 4 + 4 × 5 + 2 × 7 +
1 × 20) gives 3,460 — exactly the number of ads in Germany before any duplicates
were removed. Every original ad is accounted for, either as itself or as a copy
of a kept ad.

**Why copies are removed rather than kept:** the copies have identical texts,
so they ask for identical skills. Whether such an ad stands for one position or
several cannot be told from the data — this is true for every ad. The unit of
analysis is therefore the *job profile*, not the number of positions. This also
treats both ways of posting consistently: an employer who lists seven cities in
one ad and an employer who posts seven copies, one per city, both count once.
The number of copies is stored in `n_copies`, so the analysis can also count
every posting and check whether this changes the results.

## Off-target ads

The duplicates showed many titles that are not data roles: mechanical
engineers, safety engineers, HR managers. The search matched words like "Data"
or "Engineer" anywhere in an ad. The occupation field `hauptberuf`, assigned by
the Bundesagentur, shows which occupations the dataset actually contains.

In [ ]:
print(jobs["hauptberuf"].nunique(), "different occupations, \n" )
jobs["hauptberuf"].value_counts().head(40)

**The dataset contains 393 different occupations.** The 40 most frequent cover
2,263 ads (71%); the remaining 907 ads are spread over 353 occupations.

The list falls into three groups:

- **Clearly data roles:** Data Engineer (269), Data Scientist (213),
  Data-Analyst/in (136), Business-Intelligence-Consultant (95),
  Data-Warehouse-Analyst/in (91), KI-Engineer (83), Machine Learning Engineer (43)
  and others.
- **Clearly not data roles:** engineers in electrical engineering (68),
  aerospace (35), mechanical engineering (25) and systems engineering (25),
  sales managers.
- **Unclear:** broad occupations such as Informatiker/in (242),
  Softwareentwickler/in (155), Controller/in (80), IT-Berater/in (70) or
  Wissenschaftliche/r Mitarbeiter/in (70). A "Wissenschaftliche/r Mitarbeiter/in"
  can be a data scientist in a research project or a lab researcher; a
  "Controller/in" can work mostly with BI tools or mostly with accounting.

So `hauptberuf` alone cannot decide whether an ad is a data role. The job title
adds a second signal: does it contain a data-related term? The next cell checks
how often this is the case within each occupation.

In [ ]:
# Regular expression to identify data-related job titles: matches terms like "data", "analytics", "business intelligence", "machine learning", "KI", "AI", "statistics", and "science", but not "Datenschutz" (data protection).
DATA_TITLE = r"data|daten(?!schutz)|analy|intelligence|\bbi\b|machine learning|\bml\b|\bki\b|\bai\b|statisti|scien"

# Create a new column "data_title" that indicates whether the job title matches the data-related pattern.
jobs["data_title"] = jobs["stellenangebotsTitel"].str.contains(pat=DATA_TITLE, case=False, regex=True) # case-insensitive match, uses the DATA_TITLE regex pattern

# How many job ads have a data-related term in the title?
print(f'{jobs["data_title"].mean():.1%} of ads have a data term in the title\n')

# Analyze the share of data-related job titles within the top 40 main professions.
top = jobs["hauptberuf"].value_counts().head(40).index

# Filter the jobs to only include the top 40 main professions and then group by "hauptberuf" to calculate the number of ads and the share of data-related job titles.
(jobs[jobs["hauptberuf"].isin(top)]
 .groupby("hauptberuf")
 .agg(n_ads=("data_title", "size"), share_data_title=("data_title", "mean"))
 .sort_values("share_data_title"))

**58.7% of all ads have a data term in the title.** Within the occupations, the
share ranges from 8% to 98%, and the table sorts itself into the expected
groups:

- **Low share (below 25%):** the engineering occupations (electrical,
  mechanical, aerospace, systems), system integration specialists. These are
  mostly off-target ads. Sales managers are just above (29%).
- **High share (above 80%):** Data Scientist, Data-Analyst/in, Data Engineer,
  Machine Learning Engineer, KI-Engineer, Datenbankentwickler/in.
- **In between:** the broad occupations. 65% of the Informatiker/in ads and 41%
  of the Softwareentwickler/in ads have a data title; here the title has to
  decide.

**Neither signal is enough on its own.** Even within "Data Engineer", 19% of
the titles contain none of the terms in the pattern — either because the
pattern misses some data terms, or because the title is **too generic**. And a data
term in the title does not make a data role: a "Technical Data Engineer" in
aerospace manages technical documentation, not data pipelines. The next cell
looks at the cases where the two signals disagree. Its samples are drawn from the
first snapshot, where the title rule was developed and evaluated.

In [ ]:
# CORE: occupations that are data roles by definition
CORE = [
    "Data Engineer", "Data Scientist", "Data-Analyst/in", "Data-Warehouse-Analyst/in",
    "Machine Learning Engineer", "KI-Engineer", "KI-Manager/in", "Data-Consultant",
    "Business-Intelligence-Consultant", "Business-Analyst/in", "Datenbankentwickler/in",
    "Berufsspezialist/in - Datenanalyse",
]
# Create a new column "data_occupation" that indicates whether the main occupation is a core data role.
jobs["data_occupation"] = jobs["hauptberuf"].isin(CORE)

# Crosstab to see the relationship between data occupation and data title
print(pd.crosstab(jobs["data_occupation"], jobs["data_title"], margins=True), "\n")


# The samples are drawn from the first snapshot, where the title rule was developed
first = jobs["collected"] == pd.Timestamp("2026-10-07")

print("Data occupation, but no data term in the title:")
print(jobs.loc[first & jobs["data_occupation"] & ~jobs["data_title"], "stellenangebotsTitel"]
          .sample(15, random_state=1).to_string(), "\n") 

print("Other occupation, but a data term in the title:")
print(jobs.loc[first & ~jobs["data_occupation"] & jobs["data_title"], "stellenangebotsTitel"]
          .sample(15, random_state=1).to_string())

**The two samples point in the same direction: the job title is the more
reliable signal.**

- **Data occupation, but no data term in the title (136 ads; 78 in the first snapshot):** almost none of
  the sample are data roles: hardware and mechanical design engineers, a
  quality engineer, operations and strategy managers, a controlling student
  job. The Bundesagentur assigned them a data occupation, but the jobs
  themselves are something else. Only a few are borderline, such as an SAP
  BW/4HANA consultant (data warehousing).
- **Other occupation, but a data term in the title (917 ads; 525 in the first snapshot):** most of the
  sample are real data roles — KI-Entwickler, Data Scientist, Business
  Intelligence Developer, Cloud Data Engineer — that the Bundesagentur filed
  under broad occupations such as Informatiker/in. A few are false hits:
  "Datacenter", "Storage & Data Protection", and sales positions.

**Decision: the title decides, not the occupation.** The pattern is refined in
two ways: it also catches spellings and terms it missed so far (e.g.
"Machine-Learning", "MLOps", "NLP", "LLM", "ETL"), and it excludes the false
hits found in the sample (data centres, data protection, sales).

In [ ]:
# Modified regular expressions to identify data-related terms in job titles and to exclude false positives
DATA_TITLE = (r"data|daten(?!schutz)|analy|statisti|scien|intelligen"
              r"|\bbi\b|\bsap bw|datasphere|snowflake"
              r"|machine.?learning|maschinell|deep.?learning|computer.?vision"
              r"|\bml|mlops|\bki\b|\bai\b|genai|\bllm|\bnlp\b|prompt.?engineer|\betl\b")
NOT_DATA = r"data.?cent|rechenzentrum|data protection|data privacy|\bsales\b|vertrieb"

title = jobs["stellenangebotsTitel"]
jobs["data_role"] = (title.str.contains(DATA_TITLE, case=False, regex=True)
                     & ~title.str.contains(NOT_DATA, case=False, regex=True))

print(jobs["data_role"].value_counts())

No regex is right by construction. The standard way to test it is to label a random sample by hand and measure **precision** (how many kept ads are really data roles) and **recall** (how many data roles the rule finds).

In [ ]:
# Export a random sample for manual labeling
import html

def clean_text(s):
    """Readable ad text: decode HTML entities, drop Markdown symbols, collapse whitespace."""
       
    # Unescape Markdown symbols and clean text
    return (s.map(html.unescape)
             .str.replace(r"\\([*_])", r"\1", regex=True)               # unescape \* and \_
             .str.replace(r"\*\*|__", "", regex=True)                   # bold markers
             .str.replace(r"(?m)^[ \t]*#+[ \t]*", "", regex=True)       # heading markers at line start
             .str.replace(r"(?m)^[ \t]*[*•][ \t]+", "- ", regex=True)   # bullets → "- "
             .str.replace(r"[ \t]+", " ", regex=True)                   # collapse spaces, keep line breaks
             .str.replace(r"\n\s*\n+", "\n", regex=True)                # drop empty lines
             .str.strip())

path = PROCESSED_DIR / "label_sample.csv"
if not path.exists():
    sample = jobs.sample(100, random_state=42)[
        ["referenznummer", "stellenangebotsTitel", "hauptberuf", "stellenangebotsBeschreibung"]]
    sample["ad_text"] = clean_text(sample["stellenangebotsBeschreibung"])
    sample = sample.drop(columns="stellenangebotsBeschreibung")
    sample["role_type"] = ""   # core / adjacent / other, filled in by hand
    sample["unsure"] = ""      # 1 = hesitated
    sample.to_csv(path, index=False, encoding="utf-8-sig")
    print(len(sample), "ads exported")
else:
    print("label_sample.csv already exists - not overwritten")

Label the 100 ads in `label_sample.csv` (VS Code, "CSV" extension), using the codebook:

- `role_type`: `core`, `adjacent` or `other`, decided by the tasks in the ad text, not by the title or the official occupation.
- `unsure`: `1` if I hesitated, otherwise empty.

The sample was drawn and labelled before the second snapshot was collected, so all
100 ads come from 7 October. The cell does not overwrite the existing file, so the
labels stay valid. The two extra search terms of the second snapshot ("AI Engineer",
"MLOps") are therefore not represented in the sample.

In [ ]:
# Export 30 of the 100 sampled ads for a second annotator (blind: no labels)
second_path = PROCESSED_DIR / "label_sample_second.csv"
if not second_path.exists():
    second = (pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
              [["referenznummer", "stellenangebotsTitel", "hauptberuf", "ad_text"]]
              .sample(n=30, random_state=42))
    second["role_type"] = ""
    second["unsure"] = ""
    second.to_csv(second_path, index=False, encoding="utf-8-sig")
    print(len(second), "ads exported")
else:
    print("label_sample_second.csv already exists - not overwritten")

**Second annotator.** To check how reliable my labels are, a second person labels 30 of the 100 sampled ads with the same codebook. Both of us see exactly the same columns (title, official occupation, full ad text) and nothing else, so the second labels are independent of mine. Agreement will be measured with Cohen's kappa.

In [ ]:
# Agreement between the two annotators on the 30 shared ads
from sklearn.metrics import cohen_kappa_score

mine = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
neil = pd.read_excel(PROCESSED_DIR / "label_sample_record_neil.xlsx")

pair = mine[["referenznummer", "role_type"]].merge(
    neil[["referenznummer", "role_type"]], 
    on="referenznummer", 
    suffixes=("_me", "_neil")
)

print(len(pair), "ads labeled by both annotators\n")

print(pd.crosstab(
    pair["role_type_me"],
    pair["role_type_neil"],
    margins=True),
    "\n"
    )

# kappa on the three categories, and on the binary decision the filter actually makes
is_date_role = lambda s: s.isin(["core", "adjacent"]).astype(int)

print("kappa, three categories:", round(cohen_kappa_score(pair["role_type_me"], pair["role_type_neil"]), 3))
print("kappa, data role yes / no:", round(cohen_kappa_score(is_date_role(pair["role_type_me"]), is_date_role(pair["role_type_neil"])), 3))

### Agreement between two annotators

**Agreement is only fair: Cohen's kappa is 0.37 for the three categories and 0.35 for the binary decision (data role yes/no).** The two annotators agree on 17 of the 30 ads with three categories, and on 21 of 30 for the binary decision.

The disagreements are not random; they go in one direction:

- **The second annotator counts more ads as data roles: 24 of 30, against 17 in my labels.** Of the 13 ads I labelled *other*, he labelled 8 as data roles (5 *adjacent*, 3 *core*). The reverse happens only once.
- **Most of the difference is in *adjacent*:** he used it 13 times, I used it 6 times.

The 13 disagreements fall into three groups:

| Group | Ads | Example |
|---|---|---|
| *adjacent* vs. *other* | 5 | Cloud Platform Engineer, Finance Business Partner and Analyst |
| *core* vs. *other* | 4 | SAP Data Engineer, Tech Analyst |
| *core* vs. *adjacent* | 4 | Datenbankadministrator, Data Product Engineer |

- **The largest group points to the codebook rather than to careless labelling.** In all five cases, the second annotator chose *adjacent* for jobs where data is a tool but not the main work (cloud platforms, finance, research). The definition of *adjacent* starts with "data skills are required", which can be read as enough on its own. I applied it more narrowly, as a short list of specific job types (teaching, database administration, data quality and governance). The boundary between *adjacent* and *other* is not clear enough.
- **The *core* vs. *other* cases are genuinely ambiguous.** In three of the four, one of us marked the ad as unsure, so the `unsure` flag picks up exactly the hard cases.
- **The *core* vs. *adjacent* cases do not affect the filter,** because both count as data roles. They show that the boundary is fuzzy even for me: I labelled the database administrator as *core*, although the codebook lists database administration under *adjacent*.

Only 9 of the 13 disagreements matter for the binary decision that the filter makes.

**Decisions:**

- The kappa values are reported as they are. Relabelling the same 30 ads after seeing the second annotator's labels would make agreement look better than it is.
- My labels are used as the reference for evaluating the title rule, with this limitation in mind. With 30 ads, the kappa values are also uncertain, so my labels are one reasonable reading of the codebook, not ground truth.
- For future work, *adjacent* should be a closed list, and *other* should state explicitly: "uses data as a tool, but the job is something else (engineering, finance, project management)". A second labelling round on new ads would show whether this improves agreement.

To evaluate the title rule, classification report is used to compare the predicted labels against the ground truth labels. This helps to understand the precision, recall, and F1-score of the rule in identifying data roles correctly.

In [ ]:
from sklearn.metrics import classification_report

labels = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig").merge(
    jobs[["referenznummer", "data_role"]], on="referenznummer")

# core + adjacent = data role (1), other = not a data role (0)
labels["is_data_role"] = labels["role_type"].isin(["core", "adjacent"]).astype(int)
labels["unsure"] = labels["unsure"].fillna(0).astype(int) # convert unsure column to integer, filling NaNs with 0

y_true = labels["is_data_role"] # ground truth labels based on manual annotation
y_pred = labels["data_role"].astype(int) # predicted labels based on the title rule

print(len(labels), "labelled ads\n")
print(pd.crosstab(y_true.rename("hand label"), y_pred.rename("title rule"), margins=True), "\n")
print(classification_report(y_true, y_pred, target_names=["other", "data role"], digits=2))

### How well does the title rule work?

**The title rule is right in 80 of the 100 ads.** Compared with my hand labels:

- **44 data roles kept** and **36 non-data ads excluded** correctly.
- **12 false hits:** the rule keeps the ad, but it is not a data role.
- **8 missed data roles:** the rule excludes the ad, but it is a data role.

For the data roles, this gives a **precision of 0.79** (44 of the 56 kept ads are data roles) and a **recall of 0.85** (the rule finds 44 of the 52 data roles).

**Precision matters more here than recall.** A false hit puts a non-data job into the skill counts and distorts them. A missed data role only makes the sample a little smaller. So the 12 false hits are the main target for improving the rule.

These numbers are measured against my labels, which agreed only fairly with a second annotator (kappa 0.35 for the binary decision). They show how well the rule matches my reading of the codebook, not an absolute accuracy.

Now we check the errors made by the title rule.

In [ ]:
errors = labels[y_true != y_pred]
errors[["stellenangebotsTitel", "hauptberuf", "role_type", "unsure", "data_role"]]

### Where the title rule goes wrong

**Of the 20 errors, 12 are false hits and 8 are missed data roles.**

**Half of the false hits follow a pattern that the codebook already classifies as *other*:**

| Pattern | Ads | Matched by | Why it is *other* | Fix |
|---|---|---|---|---|
| DevOps | 3, e.g. "Senior DevOps Engineer – Schwerpunkt Big Data" | "data" | running platforms, not working with data | exclude "devops" |
| Research scientists | 2, e.g. "International Clinician Scientist" | "scien" | research in another field | drop "scien"; data scientists are already matched by "data" |
| IT infrastructure | 1, "IT AI Infrastructure Engineer" | "AI" | IT infrastructure | exclude IT infrastructure, but keep *data* infrastructure |

The other six false hits contain a real data term (e.g. "SAP Data Engineer", "Finance Business Partner and Analyst"). Excluding them would also remove real data roles, and several of them were disputed between the two annotators.

**The missed data roles cannot be fixed through the title.** Seven of the eight titles contain no data term at all (e.g. "Senior Controller (FP&A)", "Business Performance Management DACH"); only the ad text shows the data work. This is the built-in limit of a title rule, and it is acceptable because recall matters less than precision here.

**Decision: one revision of the rule.** I remove "scien" and exclude DevOps and IT infrastructure. Each change follows from the codebook, not only from these 100 ads. Still, the revised numbers are measured on the same sample that suggested the changes, so they are somewhat optimistic.

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Revised regex: "scien" removed; DevOps and IT infrastructure excluded (both "other" in the codebook)
DATA_TITLE_V2 = (r"data|daten(?!schutz)|analy|statisti|intelligen"
                 r"|\bbi\b|\bsap bw|datasphere|snowflake"
                 r"|machine.?learning|maschinell|deep.?learning|computer.?vision"
                 r"|\bml|mlops|\bki\b|\bai\b|genai|\bllm|\bnlp\b|prompt.?engineer|\betl\b")
NOT_DATA_V2 = (r"data.?cent|rechenzentrum|data protection|data privacy|\bsales\b|vertrieb"
               r"|devops|(?<!data.)(?<!daten.)infrastru")   # keeps "Data Infrastructure"

title = jobs["stellenangebotsTitel"]
jobs["data_role_v2"] = (title.str.contains(DATA_TITLE_V2, case=False, regex=True)
                        & ~title.str.contains(NOT_DATA_V2, case=False, regex=True))

labels = labels.drop(columns="data_role_v2", errors="ignore").merge(
    jobs[["referenznummer", "data_role_v2"]], on="referenznummer")
y_true = labels["is_data_role"]

compare = pd.DataFrame({
    name: {"kept": int(pred.sum()),
           "false hits": int(((pred == 1) & (y_true == 0)).sum()),
           "missed": int(((pred == 0) & (y_true == 1)).sum()),
           "precision": round(precision_score(y_true, pred), 2),
           "recall": round(recall_score(y_true, pred), 2)}
    for name, pred in [("v1", labels["data_role"].astype(int)),
                       ("v2", labels["data_role_v2"].astype(int))]})
print(compare, "\n")

print("Ads in the sample whose decision changed:")
labels.loc[labels["data_role"] != labels["data_role_v2"],
           ["stellenangebotsTitel", "role_type", "data_role", "data_role_v2"]]

**The revised rule (v2) removes exactly the six patterned false hits and nothing else in the sample:**

| | v1 | v2 |
|---|---|---|
| Ads kept | 56 | 50 |
| False hits | 12 | 6 |
| Missed data roles | 8 | 8 |
| Precision | 0.79 | 0.88 |
| Recall | 0.85 | 0.85 |

**Precision rises from 0.79 to 0.88, and recall stays at 0.85.** The six ads that changed are the three DevOps roles, the two research scientists and the IT infrastructure engineer, all labelled *other*. No ad I labelled as a data role was lost.

Because the changes were suggested by errors in this same sample, the v2 numbers are somewhat optimistic. The next check is whether v2 also removes real data roles in the rest of the dataset.

In [ ]:
dropped = jobs.loc[jobs["data_role"] & ~jobs["data_role_v2"], "stellenangebotsTitel"] 
added = jobs.loc[~jobs["data_role"] & jobs["data_role_v2"], "stellenangebotsTitel"]

print("data-role ads in the full dataset:", jobs["data_role"].sum(), "(v1) ->", jobs["data_role_v2"].sum(), "(v2)")
print(len(dropped), "removed,", len(added), "added\n")
print("A sample of removed ads:")
print(dropped.sample(min(25, len(dropped)), random_state=1).to_string())

**Checking v2 on the full dataset.** v2 keeps 1,773 data-role ads instead of 1,842: 69 are removed (38 in the first snapshot, 31 in the second), and none are added.

Most removals are correct according to the codebook:

- **Research scientists in other fields,** e.g. "Scientist LiFi / WiFi", "Principal Scientist | Target Discovery – Liver Diseases", "Research Scientist – Hyperspectral Remote Sensing"
- **DevOps roles,** e.g. "Azure DevOps Engineer mit Fokus Azure AI", "DevOps Engineer / System Engineer Linux & Data…"
- **IT and AI infrastructure,** e.g. "AI Infrastructure Engineer im Data Innovation …", "Senior AI Infrastructure & Frontier Model Engineer"

**But a few real data roles are removed by mistake:**

| Removed by mistake | Cause |
|---|---|
| "BI Engineer – Data Engineering & KI-Infrastruktur" (3 ads, one from the second snapshot) | the infrastructure exclusion |
| "Senior Applied Scientist – Demand Forecast" | dropping "scien" also loses "Applied Scientist", which is data science |

So the infrastructure exclusion is too broad when the title also names an explicit data role, and "scien" was covering a few data science titles that do not contain "data".

**The fix:** make the infrastructure exclusion **overridable**. If the title names an explicit data role (BI, data engineer, data scientist, applied scientist …), the ad is kept. The DevOps exclusion stays absolute. 

In [ ]:
# v3: the infrastructure exclusion no longer applies when the title names an explicit data role
DATA_TITLE_V3 = DATA_TITLE_V2 + r"|applied.?scien|decision.?scien"
STRONG = r"data.?(?:engineer|scien|analy)|\bbi\b|business.?intelligen|machine.?learning|applied.?scien|decision.?scien"
HARD_EXCLUDE = r"data.?cent|rechenzentrum|data protection|data privacy|datenschutz|\bsales\b|vertrieb|devops"
SOFT_EXCLUDE = r"(?<!data.)(?<!daten.)infrastru"

has = lambda pattern: title.str.contains(pattern, case=False, regex=True)
jobs["data_role_v3"] = has(DATA_TITLE_V3) & ~has(HARD_EXCLUDE) & (~has(SOFT_EXCLUDE) | has(STRONG))

# sample: should be identical to v2
labels = labels.drop(columns="data_role_v3", errors="ignore").merge(
    jobs[["referenznummer", "data_role_v3"]], on="referenznummer")
print("sample decisions changed v2 -> v3:", (labels["data_role_v2"] != labels["data_role_v3"]).sum(), "\n")

# full dataset: what v3 removes compared with v1
dropped = jobs.loc[jobs["data_role"] & ~jobs["data_role_v3"], "stellenangebotsTitel"]
print("data-role ads:", jobs["data_role"].sum(), "(v1) ->", jobs["data_role_v2"].sum(), "(v2) ->", jobs["data_role_v3"].sum(), "(v3)")
print(len(dropped), "removed compared with v1\n")
print(dropped.to_string())



**v3: a narrower exclusion.** The infrastructure exclusion now applies only when the title does not name an explicit data role (BI, data engineer, data scientist, applied scientist, machine learning). "Applied scientist" and "decision scientist" are added as data terms. The DevOps exclusion stays absolute, as the codebook classifies DevOps as *other*. In addition, the German "Datenschutz" is now excluded like the English "data protection": before, a title such as "Legal Counsel – Datenschutz & Data Governance" still got in because it also contains "Data".

| | v1 | v2 | v3 |
|---|---|---|---|
| Data-role ads in the full dataset | 1,842 | 1,773 | 1,777 |
| of these from the first snapshot | 1,104 | 1,066 | 1,068 |
| Precision in the sample | 0.79 | 0.88 | 0.88 |
| Recall in the sample | 0.85 | 0.85 | 0.85 |

**v3 brings back the wrongly removed data roles and changes nothing in the labelled sample.** Compared with v1, it removes 65 ads (36 in the first snapshot, 29 in the second), and they fit the codebook: research scientists in other fields (e.g. "Senior Scientist of Human Genetics", "Scientist LiFi / WiFi", "Senior Bioreactor Fermentation Scientist"), DevOps roles (12 of the 29 in the second snapshot, e.g. "Senior DevOps Engineer | Data Platform"), AI and IT infrastructure, one data protection role, and a dual study programme ("Bachelor of Science"). A few removals are borderline computational research roles ("Real World Evidence Scientist", "Postdoctoral Fellow in Computational N…", "Bioinformatician / Computational Scientist", "Principal Scientist – In Silico Research & Modeling") and stay out.

One ad from the second snapshot is kept by v3 although it is borderline: "HPC Systems Engineer – Machine Learning Infrastructure & Operations". The title names machine learning, so the infrastructure exclusion does not apply, but by the codebook the job is closer to IT infrastructure. One case among 1,777 does not change the results.

These adjustments came from checking the unlabelled data, not from the labels, so the sample numbers stay a fair estimate for v3 — for the first snapshot, from which the sample was drawn. **v3 is the final title rule.**

Now we apply the v3-based filter:

In [ ]:
# Final title rule: v3 (guard: only filter once)
if "data_role_v3" in jobs.columns:
    jobs["data_role"] = jobs["data_role_v3"]
    jobs = (jobs[jobs["data_role"]]
            .drop(columns=["data_title", "data_occupation", "data_role", "data_role_v2", "data_role_v3"], errors="ignore")
            .reset_index(drop=True))

# Tag adjacent roles by title, to analyse them separately later
ADJACENT = (r"dozent|trainer|lehr|teach|tutor|coach|weiterbildung|ausbild"
            r"|datenbankadmin|database admin|data quality|datenqualit|data steward"
            r"|(?:data|daten|ai|ki)[ -]?governance")
jobs["adjacent"] = jobs["stellenangebotsTitel"].str.contains(ADJACENT, case=False, regex=True)

print(jobs.shape, "\n")
print(jobs["adjacent"].value_counts(), "\n")

import re # for regular expression operations

tagged = jobs.loc[jobs["adjacent"], "stellenangebotsTitel"]
matched = tagged.str.extract(f"({ADJACENT})", flags=re.IGNORECASE)[0]
print(pd.DataFrame({"matched": matched, "title": tagged}).to_string())

In [ ]:
# Count the number of adjacent roles by group
groups = {"data quality": r"quality|qualit|steward",
          "governance": r"governance",
          "database administration": r"admin",
          "teaching / training": r"dozent|trainer|lehr|teach|tutor|coach|weiterbildung|ausbild"}
print(pd.Series({g: matched.str.contains(p, case=False).sum() for g, p in groups.items()}))

### Applying the filter

**1,777 ads remain after the final title rule (v3): 1,068 from the first snapshot and 709 from the second.** 44 of them are tagged as *adjacent*: data quality and stewardship (16), data and AI governance (19), database administration (3), and teaching or training on data and AI (6). They stay in the dataset but can be analysed separately. One tag is wrong: "Projektkoordinator*in Data Warehouse & Business Intelligence in Studium und Lehre" is matched by "Lehr" (in *Lehre*), but coordinates a university's BI projects rather than teaching.

"Governance" counts only as data or AI governance. On its own, it also matched procurement and accounting governance ("Experte Data Analytics … in der Einkaufssteuerung / Governance", "Financial Analyst … Accounting Governance"), which are not adjacent roles.

The table is saved as `data/processed/jobs_data_roles.pkl` for the extraction notebook. Pickle keeps the data types, lists and missing values exactly as they are.

In [ ]:
# Save the cleaned data-role ads for the extraction notebook
out = PROCESSED_DIR / "jobs_data_roles.pkl"
jobs.to_pickle(out)
print(jobs.shape, "->", out.resolve())
print("exists:", out.exists())

## Summary

This notebook turns the raw collection into a clean set of data-role ads.

| Step | Ads | Decision |
|---|---|---|
| Ads with full text | 3,570 | from `01_collect`: 2,119 from 7 October, 1,451 from 19 November |
| Location in Germany | 3,460 | 110 ads abroad removed (mostly Austria) |
| Exact duplicates removed | 3,257 | 203 copies, most of them ads published again between the two snapshots; unit = job profile, copies counted in `n_copies` |
| Near-duplicates removed | 3,170 | 87 copies with text similarity ≥ 0.95 |
| Data roles only (title rule v3) | **1,777** | 1,068 from the first snapshot, 709 from the second; 44 tagged as *adjacent* |

**The title rule was validated, not assumed.** On a random sample of 100 hand-labelled ads from the first snapshot, the final rule reaches a precision of 0.88 and a recall of 0.85. It was revised once based on the errors in the sample (v2) and once based on a check of the unlabelled data (v3). On the second snapshot, the same check of all removed titles found no new kind of error.

**The hand labels have a known limitation.** A second annotator labelled 30 of the ads; agreement was only fair (Cohen's kappa 0.35 for data role yes/no). Most disagreements came from the unclear boundary between *adjacent* and *other* in the codebook.

**Open points:**

- 34 near-duplicate groups with a similarity between 0.68 and 0.95 were kept without a manual check.
- The labelled sample comes from the first snapshot only; ads found by the two new search terms ("AI Engineer", "MLOps") are not part of it.
- The ad texts still contain Markdown and HTML entities; they are cleaned at the start of `03_extraction`, where they are used.

The result is saved as `data/processed/jobs_data_roles.pkl`.